In [ ]:
####################################
# imports
####################################
from pathlib import Path
import pandas as pd
import re
import unicodedata

In [ ]:
####################################
# local file paths
####################################
input_candidates = [
    Path("working_files/data/maknuune-v1.0.1.tsv"),
    Path("data/maknuune-v1.0.1.tsv"),
    Path("maknuune-v1.0.1.tsv"),
]

input_file = next((path for path in input_candidates if path.exists()), None)
if input_file is None:
    searched = "\n".join(str(path) for path in input_candidates)
    raise FileNotFoundError(f"Could not find the raw TSV file. Searched:\n{searched}")

output_file = input_file.with_name(f"{input_file.stem}_cleaned.csv")
print("Input file:", input_file)
print("Output file:", output_file)

In [ ]:
####################################
# load tsv into df
####################################
df = pd.read_csv(input_file, sep="\t")
print("Original DataFrame loaded.")
print("Shape:", df.shape)
print("Columns:", list(df.columns))

In [ ]:
####################################
# arabizi cleaning function
####################################
def clean_caphi(text):
    if pd.isna(text):
        return text
    text = str(text).lower()
    parts = text.split("#")
    cleaned_parts = []
    for part in parts:
        part = re.sub(r"[^\w\s]", "", part)
        part = part.replace(" ", "")
        if part:
            cleaned_parts.append(part)
    return " ".join(cleaned_parts)

In [ ]:
####################################
# arabic cleaning functions
####################################
ARABIC_HARAKAT = {
    "\u0610", "\u0611", "\u0612", "\u0613", "\u0614", "\u0615", "\u0616", "\u0617",
    "\u0618", "\u0619", "\u061a", "\u064b", "\u064c", "\u064d", "\u064e", "\u064f",
    "\u0650", "\u0651", "\u0652", "\u0653", "\u0654", "\u0655", "\u0656", "\u0657",
    "\u0658", "\u0659", "\u065a", "\u065b", "\u065c", "\u065d", "\u065e", "\u065f",
    "\u0670", "\u06d6", "\u06d7", "\u06d8", "\u06d9", "\u06da", "\u06db", "\u06dc",
    "\u06df", "\u06e0", "\u06e1", "\u06e2", "\u06e3", "\u06e4", "\u06e7", "\u06e8",
    "\u06ea", "\u06eb", "\u06ec", "\u06ed",
}

def clean_arabic_script(text):
    if pd.isna(text):
        return text
    text = str(text)
    text = "".join(
        ch for ch in text
        if not unicodedata.category(ch).startswith("P")
    )
    return " ".join(text.split())

def strip_arabic_harakat(text):
    if pd.isna(text):
        return text
    return "".join(ch for ch in str(text) if ch not in ARABIC_HARAKAT)

In [ ]:
####################################
# cleaning execution and testing
####################################
clean_df = pd.DataFrame({
    "arabizi": df["CAPHI++"].apply(clean_caphi),
    "arabic_harakat": df["FORM"].apply(clean_arabic_script),
})
clean_df["arabic_stripped"] = clean_df["arabic_harakat"].apply(strip_arabic_harakat)

print("\nPreview of cleaned data:")
print(clean_df.head(20))

for col in ["arabizi", "arabic_harakat", "arabic_stripped"]:
    chars = set()
    for value in clean_df[col].dropna():
        chars.update(str(value))
    print(f"\nUnique characters in {col}:")
    print(sorted(chars))

In [ ]:
####################################
# save results
####################################
clean_df.to_csv(output_file, index=False)
print("\nDone.")
print("Cleaned CSV saved to:", output_file)